- **_source_path_** — WHERE the incoming files sit. Your t_Loan parquet files. Auto Loader reads FROM here.
- **_schema_loc_** — a scratch folder where Auto Loader writes down "here are the column names and types I found." Why needed: parquet doesn't force you to declare schema upfront, so Auto Loader inspects the files once and saves the schema here to reuse next run (so it doesn't re-inspect every time).
- **_chk_loc_** — the checkpoint folder. This is the memory. Auto Loader writes here: "I processed file A, file B, file C." Next run it reads this to know what to skip.
- **_target_tbl_** — WHERE cleaned data gets written TO. Your bronze Delta table.

Think of it as: read FROM **_source_path_**, remember schema in _**schema_loc**_, remember progress in **_chk_loc_**, write TO **_target_tbl_**.

In [0]:
from pyspark.sql import functions as F

source_path = "abfss://raw@bfsilakehouse.dfs.core.windows.net/t_Loan/"  # where files land
schema_loc  = "abfss://raw@bfsilakehouse.dfs.core.windows.net/_autoloader/t_loan/schema/"   # where Auto Loader remembers the schema
chk_loc     = "abfss://raw@bfsilakehouse.dfs.core.windows.net/_autoloader/t_loan/checkpoint/"   # the checkpoint (its memory of processed files)
target_tbl  = "bfsi_lakehouse.bronze.t_loan_autoloader" # folder keeps this separate from actual data

Read this top to bottom as one sentence: "**_Set up a streaming reader, of type Auto Loader, where source files are parquet, remember schema at schema_loc, use directory-listing mode, reading from source_path._**"
Line by line:

- **_spark.readStream_** — "I want a streaming reader" (not spark.read, which is one-shot batch). Streaming = the kind that tracks progress. This is why it can be incremental.
- **_.format("cloudFiles")_** — cloudFiles is literally the codename for Auto Loader. Writing this = "use Auto Loader." If you wrote .format("parquet") instead, you'd get a plain reader with no memory.
- **_.option("cloudFiles.format", "parquet")_** — tells Auto Loader the incoming files are parquet (could be csv, json, etc). It needs to know how to open them.
- **_.option("cloudFiles.schemaLocation", schema_loc)_** — "store/read the discovered schema here." Points to your Block-1 schema folder.
- **_.option("cloudFiles.useNotifications", "false")_** — two ways Auto Loader finds new files: (a) Azure sends event notifications (needs extra permissions — the red test you saw), or (b) it lists the folder and compares to checkpoint. false = use method (b), directory listing. Simpler, works with your current permissions.
- **_.load(source_path)_** — "read from this folder."

Important: after this block, nothing has actually been read yet. df is just a recipe — "here's how to read." Spark is lazy. Reading happens only when actual write.

In [0]:
df = (spark.readStream
      .format("cloudFiles")                              # "cloudFiles" = Auto Loader
      .option("cloudFiles.format", "parquet")            # source files are parquet
      .option("cloudFiles.schemaLocation", schema_loc)   # infer + remember schema here
      .option("cloudFiles.useNotifications", "false")    # directory listing mode (no queue roles needed)
      .load(source_path))

In [0]:
df = (df
      .withColumn("_ingested_at", F.current_timestamp())    # audit columns, same as bronze
      .withColumn("_source_file", F.col("_metadata.file_path")))

Read as one sentence: "**_Write the stream as Delta, track progress at chk_loc, allow new columns, process everything available then stop, into the target table._**"
Line by line:

- **_df.writeStream_** — "write the streaming data." Pairs with readStream. This line triggers the actual work — now Spark opens files and processes.
- **_.format("delta")_** — write output as Delta table (your bronze layer format).
- **_.option("checkpointLocation", chk_loc)_** — THE key line. "Record progress here." After processing files, Auto Loader writes to this checkpoint: "done up to here." This is what makes run 2 skip everything — it reads this checkpoint first. Delete this folder and Auto Loader forgets everything and reprocesses from scratch.
- **_.option("mergeSchema", "true")_** — if a new file has an extra column, allow it into the table instead of erroring. Handles schema growth.
- **_.trigger(availableNow=True)_** — "process all files available right now, then STOP." Without this, a stream runs forever waiting for new files (real streaming). availableNow makes it behave batch-like: do current work, finish, exit. This is why your notebook cell completed instead of hanging. This is the serverless-safe, batch-style trigger.
- **_.toTable(target_tbl)_** — write into your bronze table.
- **_query.awaitTermination()_** — "wait here until the processing finishes before moving on." Without it, the cell would return immediately while work continues in background.

In [0]:
query = (df.writeStream
         .format("delta")
         .option("checkpointLocation", chk_loc)          # the memory
         .option("mergeSchema", "true")                  # allow new columns
         .trigger(availableNow=True)                     # process all available, then STOP
         .toTable(target_tbl))
query.awaitTermination()
print("✓ Auto Loader run complete")

In [0]:
%sql
SELECT count(*) FROM bfsi_lakehouse.bronze.t_loan_autoloader;
-- 1st Run - 871939
-- 2nd Run - No Changes as no new files
-- 3rd Run - 1790024

- **Block 1:** define 4 addresses (nothing runs)
- **Block 2:** build a "how to read" recipe with Auto Loader (nothing runs — lazy)
- **Block 4:** writeStream → NOW it runs:
-    1. Auto Loader lists source_path → sees all t_Loan files
-    2. Reads checkpoint (chk_loc) → "which have I done?"
-    3. First run: none done → process all → write to target table
-    4. Update checkpoint → "done files A,B,C..."
-    5. availableNow → nothing left → STOP

- 1. Lists source_path → same files (no new ones)
- 2. Reads checkpoint → "already did all of them"
- 3. Nothing new → process 0 files
- 4. Count unchanged ← the proof you saw

**The 3 concepts that make it "Auto Loader" **

- _**format("cloudFiles")**_ = it's Auto Loader.
- **_checkpointLocation_** = the memory that makes it incremental (skip done files).
- **_trigger(availableNow=True)_** = batch-style, process-then-stop (vs run-forever streaming).

The 3 ways to trigger Auto Loader (same code, different trigger)
The only thing that changes is the .trigger(...) line. Same reader, same checkpoint.
- **_1. trigger(availableNow=True) — batch style (what you used)_**

    - Process everything available now, then STOP.
    - Use for: daily/hourly scheduled runs. Job wakes, processes new files, dies.
    - Your LOS story uses this.

- **_2. trigger(processingTime="30 minutes") — micro-batch, near real-time_**

    - Job runs FOREVER. Every 30 min it wakes, grabs new files, processes, sleeps.
    - Use for: your exact example — a dashboard that must refresh every 30 min / 1 hr.
    - Cost note: cluster stays alive the whole time (it's waiting), so this costs more than batch.

- **_3. trigger(continuous=...) or default — true streaming_**

    - Processes files the moment they land, near-instant.
    - Use for: real real-time (fraud alerts, live monitoring). Rare, expensive.

**_`.trigger(processingTime="30 minutes")   # instead of availableNow=True`_**